In [1]:
import dai
import pandas as pd
import numpy as np


def main() -> pd.DataFrame:
    # ---------- 分钟级 OBI 在 SQL 里算，日频聚合也在 SQL 里做 ----------
    # 大量数据全部下推到 DAI 引擎，只把 ~250 万行日频结果带回 Python
    sql = """
    WITH minute_obi AS (
        SELECT
            date,
            instrument,
            LN(
                (bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5 + 1.0) /
                (ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5 + 1.0)
            ) AS obi
        FROM bigalpha_2026_stock_bar5m
        WHERE date >= '2019-01-01'
    ),
    daily AS (
        SELECT
            CAST(date AS DATE) AS date,
            instrument,
            AVG(obi)              AS obi_mean,
            STDDEV_SAMP(obi)      AS obi_std
        FROM minute_obi
        GROUP BY CAST(date AS DATE), instrument
    )
    SELECT
        date,
        instrument,
        obi_mean / (obi_std + 1e-6) AS factor
    FROM daily
    ORDER BY date, instrument
    """
    df = dai.query(sql).df()

    # 类型规范化
    df['date'] = pd.to_datetime(df['date'])
    df = df.dropna(subset=['factor'])
    df = df[['date', 'instrument', 'factor']]

    # ---------- 提交前自检 ----------
    assert set(df.columns) == {'date', 'instrument', 'factor'}, '列名不合规'
    daily_cov = df.groupby('date')['factor'].apply(lambda s: s.notna().mean())
    assert daily_cov.min() > 0.6, f'某日缺失率 >40%（最小覆盖 {daily_cov.min():.2%}）'

    return df

# 平台会抽取 notebook 中的代码执行，最后一格必须实际调用 main
factor_df = main()

print(f'Shape             : {factor_df.shape}')
print(f'Date range        : {factor_df["date"].min().date()} → {factor_df["date"].max().date()}')
print(f'Unique instruments: {factor_df["instrument"].nunique()}')
print(f'Factor stats      :')
print(factor_df['factor'].describe())
factor_df.head()


Shape             : (2916320, 3)
Date range        : 2019-01-02 → 2024-12-31
Unique instruments: 2252
Factor stats      :
count    2.916320e+06
mean     2.935064e-01
std      1.610290e+01
min     -4.390602e+03
25%     -1.715491e-01
50%      2.255641e-01
75%      6.964908e-01
max      2.282100e+03
Name: factor, dtype: float64


,date,instrument,factor
0,2019-01-02,000006.SZ,0.519455
1,2019-01-02,000011.SZ,0.226493
2,2019-01-02,000012.SZ,-0.902790
3,2019-01-02,000016.SZ,-0.028213
4,2019-01-02,000018.SZ,-0.492216
